# Ni X‑ray Diffraction Analysis and Rietveld Refinement

This notebook demonstrates how to process a raw Ni diffraction image (`Ni.tiff`), integrate it to a 1‑D intensity profile using **pyFAI**, and perform a **Rietveld refinement** with the **easyxrd** library.

The workflow follows the examples in the `easyXRD_examples` repository.

## Required files
* `Ni.tiff` – raw diffraction image.
* `_calibration.poni` – pyFAI calibration file.
* `_mask.edf` – bad‑pixel mask.
* `Ni.cif` – crystal structure for phase matching.
* `_instrument_parameters.gpx` – GSAS‑II instrument parameters.

All files are expected to be in the same directory as this notebook.

In [ ]:
# Install required packages (run once)
%pip install -q easyxrd pyFAI GSAS2

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from easyxrd import EasyXRD
from easyxrd.calibration import Calibration
from easyxrd.mask import Mask
from easyxrd.integration import integrate_image
from easyxrd.refinement import RietveldRefinement



Imported easyxrd with the following configuration:

easyxrd_scratch_path : /home/mt/.easyxrd_scratch
gsasii_lib_path : /home/mt/mpixi/.pixi/envs/default/lib/python3.14/site-packages/GSASII
mp_api_key : dHgNQRNYS..........


ImportError: cannot import name 'EasyXRD' from 'easyxrd' (/home/mt/mpixi/.pixi/envs/default/lib/python3.14/site-packages/easyxrd/__init__.py)

In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
# from easyxrd import EasyXRD
from easyxrd.calibration import Calibration
from easyxrd.mask import Mask
from easyxrd.integration import integrate_image
from easyxrd.refinement import RietveldRefinement

ModuleNotFoundError: No module named 'easyxrd.calibration'

In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
# from easyxrd import EasyXRD
# from easyxrd.calibration import Calibration
from easyxrd.mask import Mask
from easyxrd.integration import integrate_image
from easyxrd.refinement import RietveldRefinement

ModuleNotFoundError: No module named 'easyxrd.mask'

## 1. Load calibration and mask

In [ ]:
# Load pyFAI calibration (PONI file)
calibration_path = '_calibration.poni'
calib = Calibration.from_file(calibration_path)

# Load mask (EDf file)
mask_path = '_mask.edf'
mask = Mask.from_file(mask_path)

## 2. Load raw image and integrate to 1‑D pattern

In [ ]:
# Load the raw diffraction image
image_path = 'Ni.tiff'
# easyxrd can read many image formats via fabio
from fabio import tiffimage
raw_image = tiffimage.tiffimage(image_path).data

# Perform azimuthal integration
two_theta, intensity = integrate_image(raw_image, calib, mask)

# Plot the 1‑D pattern
plt.figure(figsize=(8,4))
plt.plot(two_theta, intensity, label='Integrated pattern')
plt.xlabel('2θ (deg)')
plt.ylabel('Intensity (a.u.)')
plt.title('Ni diffraction 1‑D profile')
plt.legend()
plt.show()

## 3. Load crystal structure (CIF) and instrument parameters

In [ ]:
# Load the CIF file for Ni phase
cif_path = 'Ni.cif'
# easyxrd can read CIF via pymatgen
from pymatgen.core import Structure
structure = Structure.from_file(cif_path)

# Load GSAS‑II instrument parameters
instr_params_path = '_instrument_parameters.gpx'
# GSASII provides a reader; easyxrd wraps it
from easyxrd.instrument import GSASIIInstrument
instrument = GSASIIInstrument.from_gpx(instr_params_path)

## 4. Set up and run Rietveld refinement

In [ ]:
# Initialise the refinement object
refiner = RietveldRefinement(
    two_theta, intensity,
    structure=structure,
    instrument=instrument,
    calibration=calib,
    mask=mask
)

# Perform the refinement (default settings)
refiner.run()

# Plot observed vs calculated pattern
refiner.plot_fit()

## 5. Save results
The refined parameters and the final pattern can be saved for further analysis.

In [ ]:
# Save the refined CIF (if lattice parameters changed)
refined_cif_path = 'Ni_refined.cif'
refiner.structure.to(fmt='cif', filename=refined_cif_path)

# Save the fitted pattern data
np.savetxt('Ni_fit.txt', np.column_stack([refiner.two_theta, refiner.intensity_observed, refiner.intensity_calculated]), header='2theta intensity_observed intensity_calculated')